In [1]:
import sys
print(sys.executable)

C:\Users\felip\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import time

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from PreProcessorDL import PreProcessorDL
from Vectorizer import Vectorizer, Word2VecEmbedding
from Trainer import Trainer
from ModelCollection import ModelCollection
from PipelineDL import PipelineDL
from CrossValidation import CrossValidation

In [3]:
## Loading data
data_loader = DataLoader(path_train="../data/labeledTrainData.csv", path_test="../data/testData.csv")
data_splitter = DataSplitter(target_column="sentiment")
train, test = data_loader.load()
train.shape, test.shape

((25000, 3), (25000, 2))

In [4]:
## splitting data
X, y = train.drop(columns=['sentiment']), train['sentiment']
print(X.shape, y.shape)
X_train, X_test, y_train, y_test = data_splitter.split(train)
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0] / (X_test.shape[0] + X_train.shape[0])

(25000, 2) (25000,)


((20000, 2), (5000, 2), (20000,), (5000,), 0.8)

## Multi-Layer Perceptron (MLP)

In [5]:
# DL Pipeline 
params = {'vectorizer_method': "tf-idf", 'model_name': "MLP", 'epochs': 20, 'batch_size': 256}
pipeline = PipelineDL(**params)
pipeline

,vectorizer_method,'tf-idf'
,model_name,'MLP'
,epochs,20
,batch_size,256
,preprocessor_type,'default'
,lowercase,None
,remove_punctuation,None
,embedding_dim,None
,window,None
,min_count,None
,max_length,None


In [6]:
# Cross-Validation
cv = CrossValidation(pipeline=clone(pipeline))

# Holdout Validation
cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
print("Holdout Validation score: ", cv_score)

## K-fold Cross-Validation 
#cv_score = cv.evaluate(X, y)
#print("K-fold CV score: ", cv_score)

Training Activated ? True
Loss after Epoch 1: 0.5325.
Loss after gradient descent: 0.001722.
Total Training time: 64.23662686347961.
Fit time: 82.40930271148682
Training Activated ? False
Prediction time: 3.40736722946167

Holdout Validation score:  0.8852


In [7]:
## Hyper-parameter tunning
param_grid = {
    "linear_layer_sizes": [(128,), (128,64), (128,64,32)],
    "lr": [1e-4, 1e-3, 1e-2],
    "vectorizer_method": ["tf-idf", "bag of words"]
}

cv_mlp_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                           reduced=False, train_size=10000, return_loss_curves=True)
cv_mlp_results

Training Activated ? True
Loss after Epoch 1: 0.6840.
Loss after gradient descent: 0.150968.
Total Training time: 61.515870571136475.
Params:  {'linear_layer_sizes': (128,), 'lr': 0.0001, 'vectorizer_method': 'tf-idf'}
Fit time: 79.81840920448303
Training Activated ? False
Prediction time: 3.2002744674682617

Training Activated ? True
Loss after Epoch 1: 0.5840.
Loss after gradient descent: 0.040028.
Total Training time: 65.42634415626526.
Params:  {'linear_layer_sizes': (128,), 'lr': 0.0001, 'vectorizer_method': 'bag of words'}
Fit time: 84.24104690551758
Training Activated ? False
Prediction time: 2.9780197143554688

Training Activated ? True
Loss after Epoch 1: 0.5285.
Loss after gradient descent: 0.001545.
Total Training time: 64.55329465866089.
Params:  {'linear_layer_sizes': (128,), 'lr': 0.001, 'vectorizer_method': 'tf-idf'}
Fit time: 83.55272054672241
Training Activated ? False
Prediction time: 3.648369789123535

Training Activated ? True
Loss after Epoch 1: 0.3683.
Loss after 

,linear_layer_sizes,lr,vectorizer_method,fit_time,pred_time,epochs,final_loss,final_validation_score,score
0,"(128,)",0.0001,tf-idf,79.818409,3.200274,20,0.150968,NaN,0.8996
1,"(128,)",0.0001,bag of words,84.241047,2.978020,20,0.040028,NaN,0.8926
6,"(128, 64)",0.0001,tf-idf,76.776127,2.973257,20,0.032411,NaN,0.8916
12,"(128, 64, 32)",0.0001,tf-idf,88.062485,3.494330,20,0.011153,NaN,0.8892
7,"(128, 64)",0.0001,bag of words,77.265333,4.101285,20,0.005475,NaN,0.8870
13,"(128, 64, 32)",0.0001,bag of words,85.088428,3.646306,20,0.002685,NaN,0.8848
2,"(128,)",0.0010,tf-idf,83.552721,3.648370,20,0.001545,NaN,0.8844
8,"(128, 64)",0.0010,tf-idf,120.784286,5.077436,20,0.000066,NaN,0.8840
3,"(128,)",0.0010,bag of words,85.200138,3.333848,20,0.000406,NaN,0.8822
15,"(128, 64, 32)",0.0010,bag of words,92.854361,3.597442,20,0.000005,NaN,0.8814


In [9]:
## Create submission using best hyper-parameters
preds = cv.pipeline.fit(X, y).predict(test)
submission = pd.DataFrame({"id": test['id'], "sentiment": preds})
submission.to_csv("../data/submission_mlp.csv", index=False)

Training Activated ? True
Loss after Epoch 1: 0.6807.
Loss after gradient descent: 0.133262.
Total Training time: 109.51986598968506.
Training Activated ? False


## Recurrent Neural Network (RNN)

In [10]:
preprocessor = PreProcessorDL(lowercase=True, remove_punctuation=True)
embedding = Word2VecEmbedding(embedding_dim=300, window=5, min_count=2, max_length=200)

start = time.time()
X_processed = preprocessor.fit_transform(X)
print("Time preprocessing:", time.time()-start)
start = time.time()
X_embbed = embedding.fit_transform(X_processed)
print("Time embedding:", time.time()-start)

Time preprocessing: 13.767788887023926
time list of sentences: 1.7254152297973633
time word2vec: 27.621439933776855
time vocabulary: 0.040462493896484375
time embedding matrix: 0.0995798110961914
time list of sentences2: 0.6230647563934326
time list of indices: 1.5396640300750732
Time embedding: 31.914077043533325


In [11]:
start = time.time()
X_processed = preprocessor.transform(X)
print("Time preprocessing:", time.time()-start)
start = time.time()
X_embbed = embedding.transform(X_processed)
print("Time embedding:", time.time()-start)

Time preprocessing: 9.426939249038696
time list of sentences2: 0.4429466724395752
time list of indices: 1.2034218311309814
Time embedding: 1.7228550910949707


In [12]:
print(embedding.embedding_matrix_.shape)
X_embbed.shape

torch.Size([51892, 300])


torch.Size([25000, 200])

In [13]:
embedding.word2vec_.wv.most_similar("good")

[('decent', 0.7383852601051331),
 ('great', 0.7018114328384399),
 ('fine', 0.6608012914657593),
 ('bad', 0.6567182540893555),
 ('cool', 0.6367418169975281),
 ('nice', 0.6289641261100769),
 ('funny', 0.5847934484481812),
 ('interesting', 0.5794842839241028),
 ('weak', 0.5773574113845825),
 ('solid', 0.5673808455467224)]

In [14]:
embedding.word2vec_.wv.vectors.shape

(51890, 300)